#Imports

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import os
import time
import datetime
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# 2 Μέρος 1: Χρήση του GPT-2

##2.1

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id
model = GPT2LMHeadModel.from_pretrained('gpt2')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(
      **encoded_text,
      max_new_tokens=200,
    do_sample=True, top_k=20)
  response_text = tokenizer.decode(response[0], skip_special_tokens=True)
  print(response_text)


##2.2 Στρατηγικές Αποκωδικοποίησης

###Greedy search

Χρησιμοποιώντας την τεχνική greedy, επιλέγεται η επόμενη λέξη με την μεγαλύτερη πιθανότητα. Παρατηρούμε το μοντέλο να επαναλαμβάνεται

In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(
      **encoded_text,
        max_new_tokens=200,
        do_sample=False)
  response_text = tokenizer.decode(response[0], skip_special_tokens=True)
  print(response_text)


###Beam Search

Χρησιμοποιώντας beam search το μοντέλο κρατάει κάποιες διαδρομές (αυτό το δέιχνει η παράμετρος num beams)  και επιλέγει την διαδρομή που μεγιστοποιεί το γινόμενο των πιθανοτήτων των λέξεων της. Δεν εγγυάται όμως ότι θα βρει την καλύτερη διαδρομή πάντα. Παρατηρούμε πάλι το κείμενο να επαναλαμβάνεται.

In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(
      **encoded_text,
        max_new_tokens=200,
        num_beams=8,
        early_stopping=True)
  response_text = tokenizer.decode(response[0], skip_special_tokens=True)
  print(response_text)

###Top-K

Ο αλγόριθμος top-k επιλέγει τυχαία από τις top-k λέξεις την επόμενη με βάση την κατανομή των πιθανοτήτων. Στο παρακάτω παράδειγμα ο αλγόριθμοςκρατάει τις top 50 λέξεις και από αυτές επιλέγει τυχαία την επόμενη.

In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(**encoded_text,max_new_tokens=200,
       do_sample=True,
      top_k=50)

  response_text = tokenizer.decode(response[0], skip_special_tokens=True)
  print(response_text)

Παρατηρούμε τώρα ότι σαν απάντηση έχουμε ένα κείμενο που μοιάζει πολύ κοντά σαν να γράφτηκε από άνθρωπο. Παρ όλα αυτά το κέιμενο αυτό δεν έχει πολύ συνοχή. Εισάγωντας όμως ένα βαθμό τυχαιότητας παίρνουμε μέχρι τώρα τα καλύτερα αποτελέσματα

###Top-p

O top-p αλγόριθμος αναζητά την καλύτερη επόμενη λέξη  τυχαία από ένα (ταξινομημένο με βάση την πιθανότητα)σύνολο λέξεων που έχει αθροιστική πιθανότητα μεγαλύτερη του . Στην παρακάτω περίπτωση αναζητούμε σε κάθε βήμα την καλύτερη επόμενη λέξη που καλύπτει σαν άθροισμα πιθανοτήτων το 95%.Εδώ δεν έχουμε σταθερό αριθμό λέξεων όπως στην περίπτωση του top-k ,αλλά το σύνολο των πιθανών επόμενων λέξεων διαφέρει ανάλογα με την πιθανότητα.

In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(**encoded_text,max_new_tokens=200,
      do_sample=True,
      top_p=0.95,
      top_k=0)

  response_text = tokenizer.decode(response[0], skip_special_tokens=True)
  print(response_text)

# 3 Μέρος 2o: Προσαρμογή Μοντέλου

##3.1

In [ ]:
torch.manual_seed(42)
class GPT2Dataset(Dataset):
  def __init__(self, txt_list, tokenizer, gpt2_type="gpt2", max_length=768):
    self.tokenizer = tokenizer
    self.input_ids = []
    self.attn_masks = []
    for txt in txt_list:
      encodings_dict = tokenizer('<|startoftext|>' + txt + '<|endoftext|>',
                                truncation=True, max_length=max_length,
                                padding="max_length")
      self.input_ids.append(torch.tensor(encodings_dict['input_ids']))
      self.attn_masks.append(torch.tensor(encodings_dict['attention_mask']))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.attn_masks[idx]

filename = 'star_wars.txt'

with open(filename, encoding='utf-8') as file:
  starwars = [line.rstrip() for line in file]

nlines = 8
min_nlines=3
l = len(starwars)
starwars_seq = []

for i in range(l-nlines):
  range_end = min(l-min_nlines,i+nlines)
  interaction = '\n'.join(starwars[i:range_end])
  starwars_seq.append(interaction)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2', bos_token='<|startoftext|>',
                                          eos_token='<|endoftext|>',
                                          pad_token='<|pad|>')
batch_size = 2

dataset = GPT2Dataset(starwars_seq, tokenizer, max_length=768)
dataloader = DataLoader(dataset, sampler=RandomSampler(dataset),
                        batch_size=batch_size)

configuration = GPT2Config.from_pretrained('gpt2', output_hidden_states=False)
model = GPT2LMHeadModel.from_pretrained("gpt2", config=configuration)
model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda")

epochs = 5
learning_rate = 5e-4
warmup_steps = 1e2
epsilon = 1e-8

sample_every = 100

optimizer = AdamW(model.parameters(),
                  lr=learning_rate,
                  eps=epsilon)

total_steps = len(dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)
total_t0 = time.time()
training_stats = []

model = model.to(device)


def format_time(elapsed):
  return str(datetime.timedelta(seconds=int(round((elapsed)))))

# Train the model
model.train()
for epoch_i in range(0, epochs):
  print("")
  print(f'======== Epoch {epoch_i + 1} / {epochs} ========')
  print('Training...')
  t0 = time.time()
  total_train_loss = 0

  for step, batch in enumerate(dataloader):
    b_input_ids = batch[0].to(device)
    b_labels = batch[0].to(device)
    b_masks = batch[1].to(device)
    model.zero_grad()
    outputs = model(b_input_ids,
                    labels=b_labels,
                    attention_mask=b_masks,
                    token_type_ids=None)
    loss = outputs[0]
    batch_loss = loss.item()
    total_train_loss += batch_loss

    # Get sample every x batches.
    if step % sample_every == 0 and not step == 0:
      elapsed = format_time(time.time() - t0)
      print(f' Batch {step:>5,} of ' +
            f'{len(dataloader):>5,}. Loss: {batch_loss:>5,}.' +
            f'Elapsed: {elapsed}.')
      model.eval()

      sample_outputs = model.generate(
                        do_sample=True,
                        top_k=50,
                        max_length=768,
                        top_p=0.95,
                        num_return_sequences=1
      )

      for i, sample_output in enumerate(sample_outputs):
        sample_output_dec = tokenizer.decode(sample_output,
                                            skip_special_tokens=True)
        print(f"{i}: {sample_output_dec}")
      model.train()
    loss.backward()
    optimizer.step()
    scheduler.step()

  # Calculate the average loss over all of the batches.
  avg_train_loss = total_train_loss / len(dataloader)
  # Measure how long this epoch took.
  training_time = format_time(time.time() - t0)
  print("")
  print(f" Average training loss: {avg_train_loss:0.2f}")
  print(f" Training epoch took: {training_time}")

print("")
print("Training complete!")
timediff = format_time(time.time()-total_t0)
print(f"Total training took {timediff} (h:mm:ss)")
# Saving best-practices: if you use default names for the model, you can reload
# it using from_pretrained()
output_dir = './model_save/'
# Create output directory if needed
if not os.path.exists(output_dir):
  os.makedirs(output_dir)

print(f"Saving model to {output_dir}")

# Save a trained model, configuration and tokenizer using `save_pretrained()`.
# They can then be reloaded using `from_pretrained()`
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)




##3.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('/content/drive/MyDrive/Colab Notebooks/model_save')
tokenizer.pad_token_id = tokenizer.eos_token_id
model = GPT2LMHeadModel.from_pretrained('/content/drive/MyDrive/Colab Notebooks/model_save')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
while True:
  text = input("Prompt: > ")
  encoded_text = tokenizer(text, return_tensors='pt')
  encoded_text = encoded_text.to(device)
  response = model.generate(**encoded_text,max_new_tokens=200,
      do_sample=True,
      top_p=0.95,
      top_k=0,
      num_return_sequences=3)

  response_text_0 = tokenizer.decode(response[0], skip_special_tokens=True)
  response_text_1 = tokenizer.decode(response[1], skip_special_tokens=True)
  response_text_2 = tokenizer.decode(response[2], skip_special_tokens=True)
  print("response_text_0:",response_text_0,)
  print("response_text_1:",response_text_1)
  print("response_text_2:",response_text_2)

##3.3

In [ ]:
# Get all of the model's parameters as a list of tuples.
params = list(model.named_parameters())
print('The GPT-2 model has {:} different named parameters.\n'.format(len(params)))

print('==== Embedding Layer ====\n')
for p in params[0:2]: #first 2 layers
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== First Transformer ====\n')
for p in params[2:14]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== Output Layer ====\n')
for p in params[-2:]: #last 2 parameters before output
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))
